In [ ]:
import pytest
import sys
import os

import numpy as np
import pandas as pd

import fine as fn
import pyomo.environ as pyomo



numberOfTimeSteps = 5
hoursPerTimeStep = 1

# Create an energy system model instance
esM = fn.EnergySystemModel(
locations={"Location"},
commodities={"electricity"},
numberOfTimeSteps=numberOfTimeSteps,
commodityUnitsDict={
    "electricity": r"kW$_{el}$"
    #, "cobalt": t
},
numberOfInvestmentPeriods=2, 
investmentPeriodInterval=5, 
hoursPerTimeStep=hoursPerTimeStep,
costUnit="1 Euro",
lengthUnit="km",
verboseLogLevel=1,
balanceLimit=None,
    )

 

# Ressourcengrenzen für alle Ressourcen
esM.resourceLimit = {
    'cobalt': 500,
    'lithium': 500 # Beispiel: 10000 als Limit für Cobalt
}


### Buy electricity at the electricity market
costs = pd.DataFrame(
    [
        np.array(
            [
                0.05,
                0.0,
                0,
                0,
                0

            ]
        ),

    ],
    index=["Location"]      
).T
revenues = pd.DataFrame(
    [
        np.array(
              [
                0.05,
                0.05,
                0.05,
                0.05,
                0.05
            ]
        ),

    ],
    index=["Location"]
).T
maxpurchase = (
    pd.DataFrame(
        [
            np.array(
                [
                    1e6,
                    1e6,
                    1e6,
                    1e6,
                    1e6,

                ]
            ),
        ],
        index=["Location"]
    ).T
    * hoursPerTimeStep
)

esM.add(
    fn.Source(
        esM=esM,
        name="Electricity market",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateMax=maxpurchase,
        commodityCostTimeSeries=costs,
        commodityRevenueTimeSeries=revenues,
    )
) 

"""
esM.add(
    fn.Storage(
        esM=esM,
        name="Li-ion batteries",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity=0.151,
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=22,
        MaterialIntensity={"cobalt": 2}
    )
)
"""

### Another Storage with a different Material Intensity


esM.add(
    fn.Sink(
        esM=esM,
        name="Cheap",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityMax=40,
        MaterialIntensity={"cobalt": 1, "lithium": 1}
    )
)


esM.add(
    fn.Storage(
        esM=esM,
        name="Li-ion batteries higher load",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityMax=50,
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity=0.0151,
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=22,
        MaterialIntensity={"cobalt": 1, "lithium": 1}
    )
)


"""
demand = (
    pd.DataFrame(
        [
            np.array(
                [
                    10.0,
                    10.0,
                ]
            ),
        ],
        index=["Location"]
    ).T
    * hoursPerTimeStep
)

"""
esM.add(
    fn.Source(
        esM=esM,
        name="Expensive",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityMax=100,
        MaterialIntensity={"lithium":3},
        
    )
)


total_time_steps = esM.totalTimeSteps

operationRateFix = {
    0: pd.DataFrame(
        data=10,  # ✅ Fixed demand
        index=total_time_steps,
        columns=["Location"]
    ),
    5: pd.DataFrame(
        data=20,  # ✅ Increased demand in second IP
        index=total_time_steps,
        columns=["Location"]
    ),
}

esM.add(
    fn.Source(
        esM=esM,
        name="Test Demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=operationRateFix,  # ✅ Force demand to be recognized
    )
)






# Modell optimieren
esM.optimize()

#print("\n🚀 Checking Available Sets in pyM:")
#print([var for var in dir(esM.pyM) if "MaterialIntensityVarSet" in var])

esM.getOptimizationSummary("StorageModel", outputLevel=0)




TypeError: Sink.__init__() got an unexpected keyword argument 'MaterialIntensity'

In [ ]:
import pytest
import sys
import os

import numpy as np
import pandas as pd

import fine as fn
import pyomo.environ as pyomo



numberOfTimeSteps = 5
hoursPerTimeStep = 1

# Create an energy system model instance
esM = fn.EnergySystemModel(
locations={"Location"},
commodities={"electricity"},
numberOfTimeSteps=numberOfTimeSteps,
commodityUnitsDict={
    "electricity": r"kW$_{el}$"
    #, "cobalt": t
},
numberOfInvestmentPeriods=2, 
investmentPeriodInterval=5, 
hoursPerTimeStep=hoursPerTimeStep,
costUnit="1 Euro",
lengthUnit="km",
verboseLogLevel=1,
balanceLimit=None,
    )


operationRate = {
    0: pd.DataFrame(
        data=10,  # ✅ Fixed demand
        index=total_time_steps,
        columns=["Location"]
    ),
    5: pd.DataFrame(
        data=20,  # ✅ Increased demand in second IP
        index=total_time_steps,
        columns=["Location"]
    ),
}

esM.add(
    fn.Source(
        esM=esM,
        name="Expensive",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateFix=operationRate,
        
    )
)


esM.add(
    fn.Sink(
        esM=esM,
        name="Test Demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=operationRate,  # ✅ Force demand to be recognized
    )
)

print("\n🔎 Checking Investment Periods:")
print("Investment Periods:", esM.investmentPeriodNames)
print("Investment Period Interval:", esM.investmentPeriodInterval)

esM.optimize()



esM.getOptimizationSummary("SourceSinkModel", outputLevel=0)


🔎 Checking Investment Periods:
Investment Periods: [0, 5]
Investment Period Interval: 5
Set parameter Threads to value 3
Set parameter QCPDual to value 1
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (linux64 - "Rocky Linux 8.8 (Green Obsidian)")



CPU model: Intel(R) Xeon(R) Gold 6154 CPU @ 3.00GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 36 physical cores, 72 logical processors, using up to 3 threads

Optimize a model with 26 rows, 28 columns and 53 nonzeros
Model fingerprint: 0xdb7ee4d6
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+01, 2e+01]
  RHS range        [0e+00, 0e+00]
Presolve removed 26 rows and 28 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  0.000000000e+00


Location
Component   Property                        Unit                    
Expensive   NPVcontribution                 [1 Euro]             0.0
            TAC                             [1 Euro/a]           0.0
            capacity                        [kW$_{el}$]          1.0
            capexCap                        [1 Euro/a]           0.0
            capexIfBuilt                    [1 Euro/a]           NaN
            commissioning                   [kW$_{el}$]          1.0
            commodCosts                     [1 Euro/a]           0.0
            commodRevenues                  [1 Euro/a]           0.0
            decommissioning                 [kW$_{el}$]          0.0
            invest                          [1 Euro]             0.0
            investLifetimeExtension         [1 Euro]               0
            isBuilt                         [-]                  NaN
            operation                       [kW$_{el}$*h/a]  87600.0
                                            [kW$_{el}$*h]       50.0
            opexCap                         [1 Euro/a]           0.0
            opexIfBuilt                     [1 Euro/a]           NaN
            opexOp                          [1 Euro/a]           0.0
            revenueLifetimeShorteningResale [1 Euro]               0
Test Demand NPVcontribution                 [1 Euro]             0.0
            TAC                             [1 Euro/a]           0.0
            capacity                        [kW$_{el}$]          NaN
            capexCap                        [1 Euro/a]           NaN
            capexIfBuilt                    [1 Euro/a]           NaN
            commissioning                   [kW$_{el}$]          NaN
            commodCosts                     [1 Euro/a]           0.0
            commodRevenues                  [1 Euro/a]           0.0
            decommissioning                 [kW$_{el}$]          NaN
            invest                          [1 Euro]             NaN
            investLifetimeExtension         [1 Euro]             NaN
            isBuilt                         [-]                  NaN
            operation                       [kW$_{el}$*h/a]  87600.0
                                            [kW$_{el}$*h]       50.0
            opexCap                         [1 Euro/a]           NaN
            opexIfBuilt                     [1 Euro/a]           NaN
            opexOp                          [1 Euro/a]           0.0
            revenueLifetimeShorteningResale [1 Euro]             NaN

In [ ]:
print("\n🔍 Checking Pyomo Model (`pyM`) Attributes:")
print(dir(esM.pyM))




🔍 Checking Pyomo Model (`pyM`) Attributes:
['ConstrBigM_srcSnk', 'ConstrCapToNbInt_srcSnk', 'ConstrCapToNbReal_srcSnk', 'ConstrCapacityDevelopment_srcSnk', 'ConstrCapacityMinDec_srcSnk', 'ConstrDesignBinFix_srcSnk', 'ConstrOperation1_srcSnk', 'ConstrOperation2_srcSnk', 'ConstrOperation3_srcSnk', 'ConstrOperation4_srcSnk', 'ConstrYearlyFullLoadHoursMax_srcSnk', 'ConstrYearlyFullLoadHoursMin_srcSnk', 'ConstrYearlyLimitation_srcSnk', 'ConstraintSharedPotentials', 'DecommConstrCapacityDevelopment_srcSnk', 'DesignLocationComponentVarSet_srcSnk', 'InitialYear_srcSnk', 'MaterialIntensityConstraint', 'MaterialIntensitySet', 'Obj', 'Skip', 'StockCommissioning_srcSnk', '_BlockData__autoslot_mappers', '_Block_reserved_words', '_ComponentDataClass', '_DEFAULT_INDEX_CHECKING_ENABLED', '_PPRINT_INDENT', '__auto_slots__', '__autoslot_mappers__', '__class__', '__contains__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__'

In [ ]:
esM.getOptimizationSummary("SourceSinkModel", outputLevel=2)

Location
Component          Property        Unit                          
Cheap              capacity        [kW$_{el}$]               40.0
                   commissioning   [kW$_{el}$]               40.0
                   operation       [kW$_{el}$*h/a]  318350.495933
                                   [kW$_{el}$*h]       181.706904
Electricity market NPVcontribution [1 Euro]         -50229.221479
                   TAC             [1 Euro/a]       -11648.363636
                   commodRevenues  [1 Euro/a]        11648.363636
                   operation       [kW$_{el}$*h/a]  232967.272727
                                   [kW$_{el}$*h]       132.972188
Expensive          capacity        [kW$_{el}$]              100.0
                   commissioning   [kW$_{el}$]              100.0
Test Demand        operation       [kW$_{el}$*h/a]        87600.0
                                   [kW$_{el}$*h]             50.0

In [ ]:
print("\n⚡ Checking Electricity Balance:")
if hasattr(esM.pyM, "balance"):
    for balance in esM.pyM.balance.items():
        print(balance)



⚡ Checking Electricity Balance:


In [ ]:
import pandas as pd

def check_operation_per_period(esM):
    operation_data = {}

    if hasattr(esM.pyM, "operation"):
        for (component_name, period, timestep), var in esM.pyM.operation.items():
            if component_name not in operation_data:
                operation_data[component_name] = [0] * len(esM.investmentPeriodNames)

            if var.value is not None:
                operation_data[component_name][esM.investmentPeriodNames.index(period)] += float(var.value)

    return pd.DataFrame(operation_data, index=[f"Investment Period {p}" for p in esM.investmentPeriodNames])

# Run check
operation_df = check_operation_per_period(esM)

print("\n⚡ Operation Per Investment Period:")
print(operation_df)



⚡ Operation Per Investment Period:
Empty DataFrame
Columns: []
Index: [Investment Period 0, Investment Period 5]


In [ ]:
import fine as fn
import pandas as pd
import numpy as np

# Define a minimal test case
numberOfTimeSteps = 5
hoursPerTimeStep = 1

# Create a minimal energy system model
esM = fn.EnergySystemModel(
    locations={"Location"},
    commodities={"electricity"},
    numberOfTimeSteps=numberOfTimeSteps,
    commodityUnitsDict={"electricity": "kW"},
    numberOfInvestmentPeriods=2,
    investmentPeriodInterval=5,
    hoursPerTimeStep=hoursPerTimeStep,
    costUnit="Euro",
    lengthUnit="km",
    verboseLogLevel=1,
)

# Define total time steps correctly
total_time_steps = esM.totalTimeSteps

# Electricity Market Source
maxpurchase = pd.DataFrame(1e6, index=total_time_steps, columns=["Location"])

esM.add(
    fn.Source(
        esM=esM,
        name="Electricity Market",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateMax=maxpurchase,
    )
)

# Define a test demand that should be active in both investment periods
operationRateFix = {
    0: pd.DataFrame(10, index=total_time_steps, columns=["Location"]),  # 10 kW in 2025
    5: pd.DataFrame(20, index=total_time_steps, columns=["Location"]),  # 20 kW in 2030
}

esM.add(
    fn.Sink(
        esM=esM,
        name="Test Demand",
        commodity="electricity",
        hasCapacityVariable=False,  # No investment variable, just fixed operation
        operationRateFix=operationRateFix,
    )
)

# Run the optimization
esM.optimize()

# Extract results
def get_operation_per_period(esM):
    operation_data = {}

    if hasattr(esM.pyM, "operation"):
        for (component_name, period, timestep), var in esM.pyM.operation.items():
            if component_name not in operation_data:
                operation_data[component_name] = [0] * len(esM.investmentPeriodNames)

            if var.value is not None:
                operation_data[component_name][esM.investmentPeriodNames.index(period)] += float(var.value)

    df = pd.DataFrame(operation_data, index=[f"Investment Period {p}" for p in esM.investmentPeriodNames])

    return df

# Print operation per period
operation_df = get_operation_per_period(esM)
print("\n⚡ Operation Per Investment Period:")
print(operation_df)


Set parameter Threads to value 3
Set parameter QCPDual to value 1
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (linux64 - "Rocky Linux 8.8 (Green Obsidian)")

CPU model: Intel(R) Xeon(R) Gold 6154 CPU @ 3.00GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 36 physical cores, 72 logical processors, using up to 3 threads

Optimize a model with 10 rows, 20 columns and 20 nonzeros
Model fingerprint: 0x12c817d2
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+01, 1e+06]
  RHS range        [0e+00, 0e+00]
Presolve removed 10 rows and 20 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  0.000000000e+00



⚡ Operation Per Investment Period:
Empty DataFrame
Columns: []
Index: [Investment Period 0, Investment Period 5]


In [ ]:
import pandas as pd

def check_operation_per_period(esM):
    """
    Extracts the operation rates per investment period.
    """
    operation_data = {}

    if hasattr(esM.pyM, "operation"):
        for (component_name, period, timestep), var in esM.pyM.operation.items():
            if component_name not in operation_data:
                operation_data[component_name] = [0] * len(esM.investmentPeriodNames)

            if var.value is not None:
                operation_data[component_name][esM.investmentPeriodNames.index(period)] += float(var.value)

    df = pd.DataFrame(operation_data, index=[f"Investment Period {p}" for p in esM.investmentPeriodNames])

    return df

# Run check
operation_df = check_operation_per_period(esM)

print("\n⚡ Operation Per Investment Period:")
print(operation_df)



⚡ Operation Per Investment Period:
Empty DataFrame
Columns: []
Index: [Investment Period 0, Investment Period 5]


In [ ]:
print("\n📋 List of Components in Model:")
for component in esM.componentNames:
    print(f"- {component}")


📋 List of Components in Model:
- Electricity Market
- Test Demand


In [ ]:
print("\n🔎 Checking Electricity Balance:")
for var in dir(esM.pyM):
    if "balance" in var.lower():
        print(f"- {var}")



🔎 Checking Electricity Balance:
- commodityBalanceConstraint


In [ ]:
print(f"Number of Investment Periods: {esM.numberOfInvestmentPeriods}")
print(f"Investment Period Names: {esM.investmentPeriodNames}")
print(f"Investment Period Interval: {esM.investmentPeriodInterval} years")


Number of Investment Periods: 2
Investment Period Names: [0, 5]
Investment Period Interval: 5 years


In [ ]:
esM.getOptimizationSummary("SourceSinkModel", outputLevel=2)

Location
Component          Property  Unit             
Electricity Market operation [kW*h/a]  87600.0
                             [kW*h]       50.0
Test Demand        operation [kW*h/a]  87600.0
                             [kW*h]       50.0

In [ ]:
# ✅ Ensure EnergySystemModel has 3 investment periods
esM = fn.EnergySystemModel(
    locations={"Location"},
    commodities={"electricity"},
    numberOfTimeSteps=numberOfTimeSteps,
    commodityUnitsDict={
        "electricity": r"kW$_{el}$"
    },
    hoursPerTimeStep=hoursPerTimeStep,
    costUnit="1 Euro",
    lengthUnit="km",
    verboseLogLevel=1,
    balanceLimit=None,
    
    numberOfInvestmentPeriods=3,  # ✅ Define 3 investment periods
    investmentPeriodInterval=5,  # ✅ Define time interval (every 5 years)
    startYear=2025  # ✅ Investment periods: 2025, 2030, 2035
)

# ✅ Check if investment periods are correctly initialized
print("Investment Periods:", esM.investmentPeriods)
print("Investment Period Names:", esM.investmentPeriodNames)

# ✅ Fix Resource Limits (use years, NOT indices)
esM.resourceLimit = {
    'cobalt': {2025: 50, 2030: 40, 2035: 30},
    'lithium': {2025: 50, 2030: 60, 2035: 70}
}

# ✅ Fix Storage Component with Multiple Investment Periods
esM.add(
    fn.Storage(
        esM=esM,
        name="Li-ion batteries higher load",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityMax={2025: 50, 2030: 40, 2035: 30},  
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity={2025: 0.0151, 2030: 0.012, 2035: 0.010},  
        opexPerCapacity={2025: 0.002, 2030: 0.0015, 2035: 0.001},
        interestRate=0.08,
        economicLifetime=22,
        MaterialIntensity={2025: {"cobalt": 1, "lithium": 1.2}, 2030: {"cobalt": 0.8, "lithium": 1.0}, 2035: {"cobalt": 0.6, "lithium": 0.8}}
    )
)

# ✅ Fix Industry Site Demand
esM.add(
    fn.Source(
        esM=esM,
        name="Expensive",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityMax={2025: 100, 2030: 80, 2035: 60},
        MaterialIntensity={2025: {"lithium": 3}, 2030: {"lithium": 2.5}, 2035: {"lithium": 2}}
    )
)

# ✅ Fix Sink (Consumption)
esM.add(
    fn.Sink(
        esM=esM,
        name="Cheap",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityMax={2025: 40, 2030: 35, 2035: 30},
        MaterialIntensity={2025: {"cobalt": 1, "lithium": 1}, 2030: {"cobalt": 0.9, "lithium": 0.8}, 2035: {"cobalt": 0.7, "lithium": 0.6}}
    )
)

#esM.optimize()


Investment Periods: [0, 1, 2]
Investment Period Names: [2025, 2030, 2035]


In [ ]:
"""
# 🔎 Verify Investment Periods
print(f"\n📌 Investment Periods: {esM.investmentPeriods}")
print(f"📌 Investment Period Names: {esM.investmentPeriodNames}")

# 🔎 Check if Pyomo Sets Exist
print("\n🚀 Checking Available Sets in pyM:")
print([var for var in dir(esM.pyM) if "Set" in var])

# 🔎 Check if Pyomo Variables Exist
print("\n🚀 Checking Available Variables in pyM:")
print([var for var in dir(esM.pyM) if "commis" in var])


# 🔎 Verify Material Intensity
print("\n🔎 Checking Material Intensity for All Components:")
for mdl in esM.componentModelingDict.values():
    for compName, comp in mdl.componentsDict.items():
        print(f"🔎 Component: {compName}, Material Intensity: {comp.MaterialIntensity}")

# 🔎 Verify Resource Limits Before Optimization
print(f"\n🔎 Resource Limits (Before Constraint Application): {esM.resourceLimit}")

# 🔎 Verify Optimization Model Variables Before Running
print("\n🚀 Checking Available Sets in pyM:")
print([var for var in dir(esM.pyM) if "MaterialIntensitySet" in var])

print("\n🚀 Checking Available Variables in pyM:")
print([var for var in dir(esM.pyM) if "commis" in var])

# 🔎 Check All Active Constraints in Pyomo
print("\n🚀 Checking Constraints in pyM:")
print([var for var in dir(esM.pyM) if "Constraint" in var])

print("\n🔎 Checking Resource Limits Before Constraint Application:")
for resource, limits in esM.resourceLimit.items():
    print(f"   {resource}: {limits}")

print("\n🚀 Checking Initial Values for commis_stor:")
for index in esM.pyM.commis_stor:
    print(f"   {index} -> {esM.pyM.commis_stor[index].value}")

print("\n🚀 Checking Initial Values for commis_srcSnk:")
for index in esM.pyM.commis_srcSnk:
    print(f"   {index} -> {esM.pyM.commis_srcSnk[index].value}")

print("\n🚀 Checking Material Intensity Constraint Expressions:")
for index in esM.pyM.MaterialIntensityConstraint:
    print(f"   {index} -> {esM.pyM.MaterialIntensityConstraint[index].expr}")

print("\n🚀 Checking MaterialIntensitySet Members:")
for index in esM.pyM.MaterialIntensitySet:
    print(f"   {index}")

print("\n🚀 Checking Components That Use commis_stor:")
for mdl in esM.componentModelingDict.values():
    for compName, comp in mdl.componentsDict.items():
        if "stor" in mdl.abbrvName:  # Only check storage components
            print(f"   {compName} -> {comp.MaterialIntensity}")

print("\n🚀 Checking MaterialIntensitySet Members:")
for index in esM.pyM.MaterialIntensitySet:
    print(f"   {index}")

print("\n🚀 Checking Material Intensity Constraint Expressions:")
for index in esM.pyM.MaterialIntensityConstraint:
    print(f"   {index} -> {esM.pyM.MaterialIntensityConstraint[index].expr}")
"""


'\n# 🔎 Verify Investment Periods\nprint(f"\n📌 Investment Periods: {esM.investmentPeriods}")\nprint(f"📌 Investment Period Names: {esM.investmentPeriodNames}")\n\n# 🔎 Check if Pyomo Sets Exist\nprint("\n🚀 Checking Available Sets in pyM:")\nprint([var for var in dir(esM.pyM) if "Set" in var])\n\n# 🔎 Check if Pyomo Variables Exist\nprint("\n🚀 Checking Available Variables in pyM:")\nprint([var for var in dir(esM.pyM) if "commis" in var])\n\n\n# 🔎 Verify Material Intensity\nprint("\n🔎 Checking Material Intensity for All Components:")\nfor mdl in esM.componentModelingDict.values():\n    for compName, comp in mdl.componentsDict.items():\n        print(f"🔎 Component: {compName}, Material Intensity: {comp.MaterialIntensity}")\n\n# 🔎 Verify Resource Limits Before Optimization\nprint(f"\n🔎 Resource Limits (Before Constraint Application): {esM.resourceLimit}")\n\n# 🔎 Verify Optimization Model Variables Before Running\nprint("\n🚀 Checking Available Sets in pyM:")\nprint([var for var in dir(esM.pyM) 

In [ ]:
"""
# ✅ Run Optimization
print("\n🚀 Running Optimization...")
esM.optimize()

print("\n🚀 Checking Available Sets in pyM:")
print([var for var in dir(esM.pyM) if "MaterialIntensitySet" in var])

print("\n🚀 Checking Available Variables in pyM:")
print([var for var in dir(esM.pyM) if "commis" in var])

# 🔎 Check If Decision Variables Have Values
for var in ['commis_stor', 'commis_srcSnk']:
    pyomo_var = getattr(esM.pyM, var, None)
    if pyomo_var:
        print(f"\n🚀 Values for {var}:")
        for index in pyomo_var:
            print(f"   {index} -> {pyomo_var[index].value}")


# 🔎 Check Optimization Summary
for ip in esM.investmentPeriodNames:
    print(f"\n📊 Optimization Summary for Investment Period {ip}")
    print(esM.getOptimizationSummary("StorageModel", outputLevel=2))
"""

'\n# ✅ Run Optimization\nprint("\n🚀 Running Optimization...")\nesM.optimize()\n\nprint("\n🚀 Checking Available Sets in pyM:")\nprint([var for var in dir(esM.pyM) if "MaterialIntensitySet" in var])\n\nprint("\n🚀 Checking Available Variables in pyM:")\nprint([var for var in dir(esM.pyM) if "commis" in var])\n\n# 🔎 Check If Decision Variables Have Values\nfor var in [\'commis_stor\', \'commis_srcSnk\']:\n    pyomo_var = getattr(esM.pyM, var, None)\n    if pyomo_var:\n        print(f"\n🚀 Values for {var}:")\n        for index in pyomo_var:\n            print(f"   {index} -> {pyomo_var[index].value}")\n\n\n# 🔎 Check Optimization Summary\nfor ip in esM.investmentPeriodNames:\n    print(f"\n📊 Optimization Summary for Investment Period {ip}")\n    print(esM.getOptimizationSummary("StorageModel", outputLevel=2))\n'

In [ ]:
"""
print("🔎 Resource Limits (Before Constraint Application):", esM.resourceLimit)

# ✅ Optimize Model
esM.optimize()

# ✅ Retrieve Optimization Summary per Investment Period
for ip in esM.investmentPeriodNames:
    print(f"\n📊 Optimization Summary for Investment Period {ip}")
    print(esM.getOptimizationSummary("StorageModel", ip=ip, outputLevel=2)) 
"""

'\nprint("🔎 Resource Limits (Before Constraint Application):", esM.resourceLimit)\n\n# ✅ Optimize Model\nesM.optimize()\n\n# ✅ Retrieve Optimization Summary per Investment Period\nfor ip in esM.investmentPeriodNames:\n    print(f"\n📊 Optimization Summary for Investment Period {ip}")\n    print(esM.getOptimizationSummary("StorageModel", ip=ip, outputLevel=2)) \n'

In [ ]:
esM.optimize(relaxIsBuiltBinary=True)

ERROR: Rule failed when generating expression for Constraint
MaterialIntensityConstraint with index ('Li-ion batteries higher load', 0,
2025): TypeError: unsupported operand type(s) for *: 'dict' and 'VarData'
ERROR: Constructing component 'MaterialIntensityConstraint' from data=None
failed:
        TypeError: unsupported operand type(s) for *: 'dict' and 'VarData'


TypeError: unsupported operand type(s) for *: 'dict' and 'VarData'

In [ ]:
print("Investment Periods:", esM.investmentPeriods)
print("Number of Investment Periods:", len(esM.investmentPeriods))

In [ ]:
esM.getOptimizationSummary("SourceSinkModel", outputLevel=2)

In [ ]:
print("\n🚀 Checking Which `commisVar` to Use for Each Component:\n")

for mdl in esM.componentModelingDict.values():
    commisVarName = f"commis_{mdl.abbrvName}"  # Get correct name (srcSnk, stor, etc.)
    
    if hasattr(esM.pyM, commisVarName):  # Check if it exists in Pyomo model
        commisVar = getattr(esM.pyM, commisVarName)
        print(f"✅ `{commisVarName}` exists in `pyM`")
    else:
        print(f"❌ `{commisVarName}` NOT FOUND in `pyM`")



In [ ]:
print("\n🚀 Checking Commissioning Values for All Components in `pyM`:\n")

for mdl in esM.componentModelingDict.values():
    commisVarName = f"commis_{mdl.abbrvName}"  # Dynamically get correct commissioning variable name

    if hasattr(esM.pyM, commisVarName):
        commisVar = getattr(esM.pyM, commisVarName)

        print(f"\n🔹 Using `{commisVarName}` for `{mdl.abbrvName}` Components:")

        for compName, comp in mdl.componentsDict.items():
            print(f"\n📌 **Component:** {compName}")

            for ip in esM.investmentPeriods:
                for loc in esM.locations:
                    # Ensure the variable exists before accessing it
                    if (loc, compName, ip) in commisVar:
                        value = commisVar[loc, compName, ip].value
                        print(f"  ✅ {compName} Commissioned at {loc}, Period {ip}: {value}")
                    else:
                        print(f"  ⚠️ No commissioning value for {compName} at {loc}, Period {ip}")

    else:
        print(f"❌ `{commisVarName}` NOT found in `pyM`!")


In [ ]:
for mdl in esM.componentModelingDict.values():
    commisVarName = f"commis_{mdl.abbrvName}"  # Dynamically get correct commissioning variable name


    commisVar = getattr(esM.pyM, commisVarName)
    for compName, comp in mdl.componentsDict.items():
        for ip in esM.investmentPeriods:
            for loc in esM.locations:
                # Ensure the variable exists before accessing it
                if (loc, compName, ip) in commisVar:
                    print(commisVar[loc, compName, ip])
                    value = commisVar[loc, compName, ip].value
                    print(f"  ✅ {compName} Commissioned at {loc}, Period {ip}: {value}")



In [ ]:
# 🚀 Find all available `commisVar` in `pyM`
available_commis_vars = [var for var in dir(esM.pyM) if var.startswith("commis_")]
print(f"✅ Available `commisVar` in `pyM`: {available_commis_vars}")

# 🚀 Dynamically match the correct `commisVar` for each component type
commisVars = {}

for mdl_name, mdl in esM.componentModelingDict.items():
    # Find the correct commisVar by checking available variables in `pyM`
    possible_commis_var = f"commis_{mdl.abbrvName}"  # Dynamically construct the name
    
    if possible_commis_var in available_commis_vars:
        commisVars[mdl_name] = getattr(esM.pyM, possible_commis_var)  # Retrieve the Pyomo variable
    else:
        raise AttributeError(f"❌ `{possible_commis_var}` not found in `pyM`. Available: {available_commis_vars}")

print(f"✅ Successfully mapped `commisVar` for each model: {commisVars}")
print(type(available_commis_vars[0]))


In [ ]:
print("\n🚀 Checking Values of `commis_stor` in `pyM`:")
commisVar = getattr(esM.pyM, "commis_stor", None)

if commisVar:
    for ip in esM.investmentPeriods:
        for loc in esM.locations:
            try:
                value = commisVar[loc, "", 0].value  # Change name as needed
                print(f"🔹 `Li-ion batteries` Commissioned at {loc}, Period {ip}: {value}")
            except KeyError:
                print(f"⚠️ No commissioning value for `Li-ion batteries` at {loc}, Period {ip}")
else:
    print("❌ `commis_stor` Not Found!")


In [ ]:
print("\n🚀 Checking Commissioning Variables in Pyomo (`commisVar`):")

for mdl in esM.componentModelingDict.values():
    for compName, comp in mdl.componentsDict.items():
        commisVarName = f"commis_{mdl.abbrvName}"
        if hasattr(esM.pyM, commisVarName):
            commisVar = getattr(esM.pyM, commisVarName)

            print(f"\n🔍 Component: {compName}")
            print(f"✅ `commisVar` Exists: {commisVarName}")
            print(type(commisVar))

            # Print Pyomo variable values safely
            for ip in esM.investmentPeriods:
                for loc in esM.locations:
                    try:
                        value = esM.pyM.commisVar[loc, compName, ip].value
                        print(f"🔹 {compName} Commissioned Amount at {loc}, {ip}: {value}")
                    except (KeyError, AttributeError):
                        print(f"⚠️ No commissioning value for {compName} at {loc}, {ip}")
        else:
            print(f"❌ `commisVar` NOT FOUND for {compName}!")


In [ ]:
# Wichtig

In [ ]:
for mdl in esM.componentModelingDict.values():
    for compName, comp in mdl.componentsDict.items():
        print(f" Component Name: {compName}")
        print(f"📌 MaterialIntensity: {comp.MaterialIntensity}")
        
    

In [ ]:
for mdl in esM.componentModelingDict.values():
    for compName, comp in mdl.componentsDict.items():
        print(f" Component Name: {compName}")
        print(f"📌 MaterialIntensity: {comp.MaterialIntensity}")
        print(f"commis_{mdl.abbrvName}")

In [ ]:
commisVars = {}
for mdl_name, mdl in esM.componentModelingDict.items():
    # Find the correct commisVar by checking available variables in `pyM`
    possible_commis_var = f"commis_{mdl.abbrvName}"  # Dynamically construct the name
    commisVars[mdl_name] = getattr(esM.pyM, possible_commis_var)  # Retrieve the Pyomo variable
    print(commisVars[mdl_name])

In [ ]:
print("\n🚀 Checking Commissioning Values for Each Component:")

for mdl in esM.componentModelingDict.values():
    for compName, comp in mdl.componentsDict.items():
        commisVarName = f"commis_{mdl.abbrvName}"  # Construct commisVar name dynamically

        print(f"\n🔍 Component: {compName}")
        print(f"📌 MaterialIntensity: {comp.MaterialIntensity}")

        if hasattr(esM.pyM, commisVarName):
            commisVar = getattr(esM.pyM, commisVarName)  # Get the actual Pyomo variable
            print(f"✅ `commisVar` Found: {commisVarName}")

            # 🚀 Print commissioning values for each location & investment period
            for ip in esM.investmentPeriods:
                for loc in esM.locations:
                    try:
                        print(f"🔹 {compName} Commissioned Amount at {loc}, {ip}: {pyomo.value(commisVar[loc, compName, ip])}")
                    except KeyError:
                        print(f"⚠️ {compName} Missing commissioning value at {loc}, {ip}")
        else:
            print(f"❌ ERROR: `{commisVarName}` NOT FOUND in `pyM`!")


In [ ]:
print("\n🚀 Checking If `commisVar` Exists in `pyM`:")
print(hasattr(esM.pyM, "commisVar"))  # Should return True
print("Available variables in pyM:", [var for var in dir(esM.pyM) if "commis" in var])


In [ ]:
print("\n🚀 Checking If `MaterialIntensity` Exists in `esM`:")
print(hasattr(esM, "MaterialIntensity"))  # Should return True
print("Value of esM.MaterialIntensity:", getattr(esM, "MaterialIntensity", "❌ NOT FOUND"))


In [ ]:
print("\n🚀 Checking How `investmentPeriods` Is Used Elsewhere in `esM`:")
for attr in dir(esM):
    if "investment" in attr.lower():
        print(attr, ":", getattr(esM, attr))



In [ ]:
print("\n🚀 Checking If `abbrvName` Exists in StorageModel:")
print(hasattr(esM.componentModelingDict['SourceSinkModel'], "abbrvName"))

In [ ]:
print("\n🚀 Checking `MaterialIntensityVarSet` Contents:")

# Check for StorageModel
if hasattr(esM.pyM, "MaterialIntensityVarSetstor"):
    print("\n📦 `MaterialIntensityVarSetstor` (Storage Components):")
    print(list(esM.pyM.MaterialIntensityVarSetstor))
else:
    print("\n⚠️ `MaterialIntensityVarSetstor` is missing!")

# Check for Source/Sink Components
if hasattr(esM.pyM, "MaterialIntensityVarSetsrcSnk"):
    print("\n⚡ `MaterialIntensityVarSetsrcSnk` (Source/Sink Components):")
    print(list(esM.pyM.MaterialIntensityVarSetsrcSnk))
else:
    print("\n⚠️ `MaterialIntensityVarSetsrcSnk` is missing!")


In [ ]:
esM.componentModelingDict['StorageModel'].declareMaterialIntensityVarSet(esM.pyM, esM)

# Now check if the set exists in `pyM`
print("\n🚀 Checking If `MaterialIntensityVarSet` Now Exists in `pyM`:")
print([var for var in dir(esM.pyM) if "MaterialIntensityVarSet" in var])


In [ ]:
print("\n🚀 Checking Which Components Have `MaterialIntensity` Defined:")
for compName, comp in esM.componentModelingDict['StorageModel'].componentsDict.items():
    print(f"🔍 {compName}: {comp.MaterialIntensity}")
